# Advanced retrieval comparison — Qwen 7B

This is a separate long-running experiment. It compares the existing pipeline with hybrid BM25+semantic retrieval, cross-encoder reranking, and an evidence abstention gate. It trains a fresh refusal-aware adapter and runs all five conditions into `results/advanced_7b`. Keep the T4 runtime awake and run cells in order.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Select a T4 GPU'
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, json, shutil
drive.mount('/content/drive')
DRIVE_ROOT=Path('/content/drive/MyDrive/farmer-llm-advanced-7b')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
REPO=Path('/content/farmer-llm-project')
URL='https://github.com/sarthakmehra02/stuff.git'
if not REPO.exists(): subprocess.run(['git','clone',URL,str(REPO)],check=True)
else: subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
os.chdir(REPO)
os.environ['FARMER_MODEL_ID']='Qwen/Qwen2.5-7B-Instruct'
print('Model:',os.environ['FARMER_MODEL_ID'])

In [ ]:
%pip install -r requirements-colab.txt
from config import MODEL_ID
assert MODEL_ID == 'Qwen/Qwen2.5-7B-Instruct'
print(MODEL_ID)

In [ ]:
from src.rag import build_index
index,chunks=build_index()
print('Index:',index.ntotal,'chunks:',len(chunks))

In [ ]:
from src.hybrid import HybridRetriever
from src.rag import retrieve
hybrid=HybridRetriever(candidate_k=20, top_k=4, min_rerank_score=0.5)
def reranked(question, k=4):
    return retrieve(question, k=k, rerank=True, candidate_k=20)
print('Three retrieval arms ready: semantic, reranked semantic, and hybrid gated')

## Build refusal-aware training data

These synthetic out-of-scope questions are not copied from the test set. They teach the adapter to abstain when the corpus does not contain an answer.

In [ ]:
base=Path('data/train_qa.jsonl').read_text(encoding='utf-8').splitlines()
templates=[
 'What is the 2035 PM-KISAN benefit for farmers in Kerala?',
 'Does the documents corpus state the dairy subsidy in Assam for 2031?',
 'What is the tractor subsidy under the Maharashtra scheme in 2030?',
 'Which PM-KISAN office handles complaints in an unnamed foreign country?',
 'What is the PMFBY premium for a crop and district not listed in the documents?',
 'What is the KCC interest rate for a bank and year absent from these documents?'
]
refusals=[]
for i in range(60):
    refusals.append({'question':templates[i%len(templates)] + f' Case {i+1}.','answer':'Not found in the documents','chunk_id':f'synthetic_refusal_{i+1}','source_file':'','page':0,'scheme':'out_of_scope','style':'refusal','evidence_quote':''})
Path('data/advanced_train_qa.jsonl').write_text('\n'.join(base+[json.dumps(r,ensure_ascii=False) for r in refusals])+'\n',encoding='utf-8')
print('Training rows:',len(base)+len(refusals),'refusals:',len(refusals))

In [ ]:
!python src/split_train.py --input data/advanced_train_qa.jsonl --out-dir data/advanced_train_split --val-fraction 0.15

## Run base 7B comparison

This runs zero-shot, prompted, and hybrid-gated RAG for all 60 questions.

In [ ]:
from src.models import load_model, generate
from src.run_conditions import run_conditions
from pathlib import Path
OUT_SEM=Path('results/advanced_7b/semantic')
OUT_RERANK=Path('results/advanced_7b/reranked')
OUT_HYBRID=Path('results/advanced_7b/hybrid_gated')
model,tokenizer=load_model()
for out,retriever in [(OUT_SEM,retrieve),(OUT_RERANK,reranked),(OUT_HYBRID,hybrid)]:
    run_conditions(['zero_shot','prompted','rag'], out=out, model=model, tokenizer=tokenizer, retrieve_fn=retriever, generate_fn=generate)
print('Semantic, reranked, and hybrid-gated base results complete')

## Train the refusal-aware 7B adapter

This can take 1–3 hours on a T4. The adapter is saved directly to Drive.

In [ ]:
!python src/finetune.py --train data/advanced_train_split/train.jsonl --val data/advanced_train_split/val.jsonl --out /content/drive/MyDrive/farmer-llm-advanced-7b/adapter_run --epochs 3 --max-seq-length 1024 --learning-rate 1e-4

## Run fine-tuned hybrid conditions

The same hybrid retriever and abstention gate are used, so the comparison isolates the effect of the refusal-aware adapter.

In [ ]:
del model
torch.cuda.empty_cache()
adapter='/content/drive/MyDrive/farmer-llm-advanced-7b/adapter_run/adapter'
model,tokenizer=load_model(adapter_path=adapter)
OUT_FT=Path('results/advanced_7b/finetuned_hybrid')
run_conditions(['finetuned','finetuned_rag'], out=OUT_FT, model=model, tokenizer=tokenizer, retrieve_fn=hybrid, generate_fn=generate)
for folder in [OUT_SEM,OUT_RERANK,OUT_HYBRID,OUT_FT]:
    drive_folder=DRIVE_ROOT/folder.name
    drive_folder.mkdir(parents=True,exist_ok=True)
    for p in folder.glob('raw_*.jsonl'): shutil.copy2(p,drive_folder/p.name)
    if (folder/'run_metadata.json').exists(): shutil.copy2(folder/'run_metadata.json',drive_folder/'run_metadata.json')
shutil.copy2('/content/drive/MyDrive/farmer-llm-advanced-7b/adapter_run/training_summary.json',DRIVE_ROOT/'training_summary.json')
print('All advanced outputs copied to',DRIVE_ROOT)

In [ ]:
for folder in [OUT_SEM,OUT_RERANK,OUT_HYBRID,OUT_FT]:
    print('\n',folder)
    for p in sorted(folder.glob('raw_*.jsonl')):
        n=len([x for x in p.read_text(encoding='utf-8').splitlines() if x.strip()])
        print(p.name,n,'rows')